# ISLR Pipeline — LIBRAS Landmark Subset
Pipeline completo: extração → filtro → treino → resultados

## 0. Setup

In [ ]:
import os, sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

print('Raiz do projeto:', PROJECT_ROOT)

In [ ]:
%pip install -r requirements.txt
# %pip install --upgrade torch torchvision

# PYTORCH_ENABLE_MPS_FALLBACK=1 python scripts/03_train.py ...


In [ ]:
# ── Configuração central — edite aqui ─────────────────────────────────────

DATASET      = 'include50'          # 'minds' | 'ufop' | 'ksl' | 'include50'
SUBSET       = '2nd'          # 'all' | '1st' | '2nd' | 'laines' | 'arcanjo'
EXTRACTOR    = 'mediapipe'    # 'mediapipe' | 'openpose'
MODEL        = 'resnet18'     # 'resnet18' | 'resnet50' | 'efficientnet_b6' | 'vit_medium'
IMAGE_METHOD = 'Skeleton-DML' # 'Skeleton-DML' | 'Skeleton-Magnitude' | 'SL-DML'
IMPUTE       = True
REF          = 'include50_2nd_com_imputacao' # referência para salvar os resultados (nome do experimento)

LEARNING_RATE = 0.0001
WEIGHT_DECAY  = 0.0001
EPOCHS        = 30 #50
SEED          = 42
PATIENCE      = 5

RAW_DIR       = 'data/raw'
INTERIM_DIR   = 'data/interim'
PROCESSED_DIR = 'data/processed'
OUTPUT_DIR    = 'experiments'
TABLE_DIR     = 'reports/tables'

# Datasets que usam CSV direto (sem extração de vídeo)
CSV_DATASETS = {'ksl', 'include50'}

print('Config ok')

---
## 1. Extração de landmarks
Pule se já tiver o CSV em `data/interim/<dataset>/<dataset>_mediapipe.csv`.

In [ ]:
from datasets.base_dataset import BaseVideoDataset
from extraction.base_extractor import BaseExtractor

interim_csv = os.path.join(INTERIM_DIR, DATASET, f'{DATASET}_{EXTRACTOR}.csv')

if os.path.exists(interim_csv):
    print(f'CSV já existe: {interim_csv}\nDelete o arquivo se quiser re-extrair.')
else:
    dataset_instance = BaseVideoDataset.create(DATASET, RAW_DIR)
    extractor        = BaseExtractor.create(EXTRACTOR)
    processor        = dataset_instance.get_processor(extractor)
    videos           = dataset_instance.prepare_data()

    print(f'{len(videos)} vídeos encontrados')
    os.makedirs(os.path.join(INTERIM_DIR, DATASET), exist_ok=True)
    processor.process_all(videos, interim_csv, chunk_size=10_000)
    print(f'Extração concluída → {interim_csv}')

---
## 2. Filtro + imputação
Pule se já tiver o CSV em `data/processed/<dataset>/<dataset>_<subset>.csv`.

In [ ]:
from preprocessing.filter_landmarks import filter_and_save

suffix        = '' if IMPUTE else '_no_imputation'
processed_csv = os.path.join(PROCESSED_DIR, DATASET, f'{DATASET}_{SUBSET}{suffix}.csv')

if os.path.exists(processed_csv):
    print(f'CSV já existe: {processed_csv}\nDelete o arquivo se quiser re-filtrar.')
else:
    filter_and_save(
        input_path   = interim_csv,
        output_path  = processed_csv,
        subset_name  = SUBSET,
        impute       = IMPUTE,
        dataset_name = DATASET,
    )
    print(f'Filtro concluído → {processed_csv}')

In [ ]:
# Carrega e inspeciona o DataFrame final
import pandas as pd
from datasets.base_dataset import BaseCsvDataset

if DATASET in CSV_DATASETS:
    loader = BaseCsvDataset.create(DATASET, INTERIM_DIR)
    df = loader.load()
else:
    df = pd.read_csv(processed_csv)

print(f'Shape:       {df.shape}')
print(f'Vídeos:      {df["video_name"].nunique()}')
print(f'Pessoas:     {sorted(df["person"].unique())}')
print(f'Categorias:  {df["category"].nunique()}')
df.head(15)

---
## 3. Treino LOPO
### 3a. Fold único

In [ ]:
import gc

# 1. Converte pra float32 (metade da memória)
landmark_cols = [c for c in df.columns if c.endswith(('_x', '_y', '_z'))]

# df[landmark_cols] = df[landmark_cols].astype('float32')

# INCLUDE 50
# Converte com tolerância a strings inválidas (ex.: 'True'/'False')
df[landmark_cols] = (
    df[landmark_cols]
    .replace({'True': 1.0, 'False': 0.0})
    .apply(pd.to_numeric, errors='coerce')
    .astype('float32')
)

# 2. Mantém só o necessário
keep = ['category', 'video_name', 'frame', 'person'] + landmark_cols
df = df[[c for c in keep if c in df.columns]].copy()

# 3. Força limpeza
gc.collect()

print(f'Memória do df: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

In [ ]:
from training.trainer import Trainer

# VALIDATE_PEOPLE = [1]
# TEST_PEOPLE     = [0]
VALIDATE_PEOPLE = [0, 1, 2, 3]
TEST_PEOPLE     = [0, 1, 2, 3]

result = Trainer({
    'dataset_name':    DATASET,
    'ref':             REF,
    'seed':            SEED,
    'validate_people': VALIDATE_PEOPLE,
    'test_people':     TEST_PEOPLE,
    'learning_rate':   LEARNING_RATE,
    'weight_decay':    WEIGHT_DECAY,
    'image_method':    IMAGE_METHOD,
    'model':           MODEL,
    'epochs':          EPOCHS,
    'output_dir':      OUTPUT_DIR,
    'augment_cfg': {
        'rotation_sigma':       12,
        'zoom_sigma':           0.1,
        'translate_x_sigma':    0.06,
        'horizontal_flip_prob': 0.3,
    },
}).run(df)

print(f"acc={result['test_accuracy']:.4f}  f1={result['test_f1']:.4f}")

In [ ]:
print(f"Colunas _x: {len([c for c in df.columns if c.endswith('_x')])}")
print(f"Categorias: {df['category'].nunique()}")
print(f"Vídeos por person: {df.groupby('person')['video_name'].nunique()}")

In [ ]:
print(f"Treino: {df[~df['person'].isin([0, 1])]['video_name'].nunique()} vídeos")
print(f"Val:    {df[df['person'] == 1]['video_name'].nunique()} vídeos")
print(f"Teste:  {df[df['person'] == 0]['video_name'].nunique()} vídeos")

### 3b. LOPO completo (todos os folds)

In [ ]:
from training.trainer import Trainer

LOPO_FOLDS = {
    'minds':     [(i % 12 + 1, i + 1) for i in range(12)],
    'ufop':      [(1,2),(2,3),(3,4),(4,5),(5,1)],
    'ksl': [
        ([0,1,2,3],   [0,1,2,3]),
        ([4,5,6,7],   [4,5,6,7]),
        ([8,9,10,11], [8,9,10,11]),
        ([12,13,14,15],[12,13,14,15]),
        ([16,17,18,19],[16,17,18,19]),
    ],
    # 'ksl':       [(i % 20 + 1, i) for i in range(20)],
    'include50': [(i % 22 + 1, i) for i in range(22)],
}

BASE_CFG = {
    'dataset_name':  DATASET,
    'ref':           REF,
    'seed':          SEED,
    'learning_rate': LEARNING_RATE,
    'weight_decay':  WEIGHT_DECAY,
    'image_method':  IMAGE_METHOD,
    'model':         MODEL,
    'epochs':        EPOCHS,
    'output_dir':    OUTPUT_DIR,
    'augment_cfg': {
        'rotation_sigma':       12,
        'zoom_sigma':           0.1,
        'translate_x_sigma':    0.06,
        'horizontal_flip_prob': 0.5,
    },
}

folds   = LOPO_FOLDS[DATASET]
results = []

for i, (val, test) in enumerate(folds):
    val_list  = val  if isinstance(val,  list) else [val]
    test_list = test if isinstance(test, list) else [test]
    print(f'\nFold {i+1}/{len(folds)} — val={val_list} test={test_list}')

    r = Trainer({**BASE_CFG,
                 'validate_people': val_list,
                 'test_people':     test_list}).run(df)
    results.append(r)
    print(f'✓ acc={r["test_accuracy"]:.4f}  f1={r["test_f1"]:.4f}')

print(f'\n{len(folds)} folds concluídos.')

---
## 4. Resultados

In [ ]:
import json, numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

results_path = os.path.join(OUTPUT_DIR, REF, DATASET)
loaded = [
    json.load(open(os.path.join(results_path, f)))
    for f in sorted(os.listdir(results_path))
    if f.endswith('.json')
]

print(f'{DATASET} #{REF} — {len(loaded)} folds')
print(f'model: {loaded[0].get("model")} | method: {loaded[0].get("image_method")}\n')

METRICS = [
    ('Accuracy',  accuracy_score,  {}),
    ('Precision', precision_score, {'average':'macro','zero_division':0}),
    ('Recall',    recall_score,    {'average':'macro','zero_division':0}),
    ('F1',        f1_score,        {'average':'macro','zero_division':0}),
]

summary = {}
for name, fn, kw in METRICS:
    vals = [fn(r['true_labels'], r['predicted_labels'], **kw) for r in loaded]
    summary[name] = {'mean': np.mean(vals), 'std': np.std(vals), 'folds': vals}
    print(f'  {name:<12} {np.mean(vals):.4f} ± {np.std(vals):.4f}')

In [ ]:
# Salva tabela CSV para dissertação
import pandas as pd

os.makedirs(TABLE_DIR, exist_ok=True)
table_path = os.path.join(TABLE_DIR, f'results_{DATASET}_{SUBSET}_ref{REF}.csv')

row = {'dataset': DATASET, 'subset': SUBSET, 'ref': REF,
       'model': loaded[0].get('model'), 'image_method': loaded[0].get('image_method'),
       'folds': len(loaded)}
for name, vals in summary.items():
    row[name]          = round(vals['mean'], 4)
    row[f'{name}_std'] = round(vals['std'],  4)

pd.DataFrame([row]).to_csv(table_path, index=False)
print(f'Tabela salva → {table_path}')

---
## 5. Benchmark de latência

In [ ]:
import cv2
from time import time
from tqdm.notebook import tqdm
from datasets.base_dataset import BaseVideoDataset
from extraction.base_extractor import BaseExtractor

BENCH_EXTRACTOR = 'mediapipe'
N_VIDEOS        = 5

if DATASET in CSV_DATASETS:
    print(f"Benchmark de latência não aplicável para '{DATASET}' (dataset CSV pré-processado).")
else:
    def video_frames(path):
        cap = cv2.VideoCapture(path)
        n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        return n

    dataset_instance = BaseVideoDataset.create(DATASET, RAW_DIR)
    extractor        = BaseExtractor.create(BENCH_EXTRACTOR)
    videos           = dataset_instance.prepare_data()
    sample           = sorted(videos, key=lambda v: video_frames(v[0]))
    mid              = len(sample) // 2
    sample           = sample[:N_VIDEOS] + sample[mid:mid+N_VIDEOS] + sample[-N_VIDEOS:]

    total_time, total_frames = 0.0, 0
    for video_path, *_ in tqdm(sample):
        n  = video_frames(video_path)
        t0 = time()
        for _ in extractor.get_video_landmarks(video_path):
            pass
        total_time   += time() - t0
        total_frames += n

    print(f'Extractor:   {BENCH_EXTRACTOR}')
    print(f'FPS médio:   {total_frames / total_time:.2f}')
    print(f'Tempo/vídeo: {total_time / len(sample):.3f}s')